# Fig. S2F: cross-species, cross-lineage and W-vs-expression panels

Sub-panels under `Fig2/supp_4/<panel>/`. S4a mouse vs human-organoid rho_W for orthologs
(per lineage); S4b cross-lineage EXC vs INH pseudotime Delta-psi; S4c per-gene W-vs-expression
coupling (rho), per lineage; S4e cross-species neuron-bridge per-gene scatter; S4f mouse vs
organoid Delta-psi (mouse neuron-extended trajectory). (rho_W / W-vs-expression are effect-size
descriptors; no per-gene p-values cited.)


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig2/supp_4"
KB = Path(f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling")
LINS = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]
EXC_C, INH_C = (
    S.paper_palette_lineage["excitatory"],
    S.paper_palette_lineage["inhibitory"],
)

In [ ]:
# S4a - cross-species rho_W concordance (model-independent corroboration: overall W-vs-pseudotime
# trend correlates across species better than the endpoint Delta-psi, esp. in the inhibitory lineage)
ob = pd.read_csv(KB / "orthologs_results" / "orthologs_significant_both.csv")

# top volcano hits per lineage (same criterion as fig2c: padj<0.05, |rho_W|>0.2, top-4 composite)
_comp = {
    lin: pd.read_csv(
        KB / "composite_ranking" / f"composite_ranking_optimized2_{lin}.csv"
    )
    for lin, _ in LINS
}
_top = {
    lin: set(
        _comp[lin][
            (_comp[lin]["padj"] < 0.05) & (_comp[lin]["rho_W"].abs() > 0.2)
        ].nlargest(4, "composite_score_abs")["gene_name"]
    )
    for lin, _ in LINS
}

fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, 760 / 300))
for ax, (lin, lab) in zip(axes, LINS):
    d = ob[ob.lineage == lin].dropna(subset=["rho_W_mouse", "rho_W_kn"])
    cc = np.sign(d.rho_W_mouse) == np.sign(d.rho_W_kn)
    ax.scatter(
        d.rho_W_mouse[cc], d.rho_W_kn[cc], s=9, c="#1b7837", alpha=0.7, linewidths=0
    )
    ax.scatter(
        d.rho_W_mouse[~cc], d.rho_W_kn[~cc], s=9, c="#cccccc", alpha=0.6, linewidths=0
    )
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    sr = spearmanr(d.rho_W_mouse, d.rho_W_kn)[0]
    pr = pearsonr(d.rho_W_mouse, d.rho_W_kn)[0]
    ax.set_xlabel(r"mouse SVZ  $\rho_W$", fontsize=S.FONT_LABEL)
    ax.set_ylabel(r"organoid  $\rho_W$", fontsize=S.FONT_LABEL)
    ax.set_title(
        f"{lab}: Spearman r={sr:+.2f}, Pearson={pr:+.2f} (n={len(d)})",
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        pad=3,
    )
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
    hits = d[d["human_name"].isin(_top[lin])]
    for _, r in hits.iterrows():
        ax.scatter(
            r.rho_W_mouse,
            r.rho_W_kn,
            s=20,
            c="#1b7837",
            linewidths=0.6,
            edgecolors="k",
            zorder=5,
        )
        ax.annotate(
            str(r.human_name),
            (r.rho_W_mouse, r.rho_W_kn),
            fontsize=S.FONT_ANNOT - 1,
            fontstyle="italic",
            xytext=(3, 3),
            textcoords="offset points",
        )
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.14, wspace=0.28)
S.save_and_show(fig, "fig_s4a_xspecies_rhoW", figure=FIG)

In [ ]:
# S4f - cross-species Delta-psi (mouse qNSC->neuron neuron-extended trajectory) vs organoid Delta-psi
from pathlib import Path as _P

NB_ = _P(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_neuron_bridge"
)
nbo = pd.read_csv(NB_ / "cross_species_neuron_orthologs.csv")
qn = pd.read_csv(NB_ / "cross_species_neuron_concordance.csv")
qn = qn[qn.mouse_contrast == "qNSC->neuron"].set_index("organoid_lineage")

# top volcano hits per lineage (same criterion as fig2c: padj<0.05, |rho_W|>0.2, top-4 composite)
_comp = {
    lin: pd.read_csv(
        KB / "composite_ranking" / f"composite_ranking_optimized2_{lin}.csv"
    )
    for lin, _ in LINS
}
_top = {
    lin: set(
        _comp[lin][
            (_comp[lin]["padj"] < 0.05) & (_comp[lin]["rho_W"].abs() > 0.2)
        ].nlargest(4, "composite_score_abs")["gene_name"]
    )
    for lin, _ in LINS
}

fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, 760 / 300))
for ax, (lin, lab, ocol) in zip(
    axes,
    [
        ("excitatory", "Excitatory neurons", "dpsi_org_exc"),
        ("inhibitory", "Inhibitory neurons", "dpsi_org_inh"),
    ],
):
    d = nbo.dropna(subset=["dpsi_qn", ocol])
    cc = np.sign(d.dpsi_qn) == np.sign(d[ocol])
    ax.scatter(d.dpsi_qn[cc], d[ocol][cc], s=9, c="#1b7837", alpha=0.7, linewidths=0)
    ax.scatter(d.dpsi_qn[~cc], d[ocol][~cc], s=9, c="#cccccc", alpha=0.6, linewidths=0)
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    sr = qn.loc[lin, "spearman"]
    ng = int(qn.loc[lin, "n_genes"])
    ax.set_xlabel(r"mouse qNSC$\rightarrow$neuron  $\Delta\psi$", fontsize=S.FONT_LABEL)
    ax.set_ylabel(r"organoid  $\Delta\psi$", fontsize=S.FONT_LABEL)
    ax.set_title(
        f"{lab}: Spearman r={sr:+.2f} (n={ng})",
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        pad=3,
    )
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
    hits = d[d["human_name"].isin(_top[lin])]
    for _, r in hits.iterrows():
        ax.scatter(
            r.dpsi_qn,
            r[ocol],
            s=20,
            c="#1b7837",
            linewidths=0.6,
            edgecolors="k",
            zorder=5,
        )
        ax.annotate(
            str(r.human_name),
            (r.dpsi_qn, r[ocol]),
            fontsize=S.FONT_ANNOT - 1,
            fontstyle="italic",
            xytext=(3, 3),
            textcoords="offset points",
        )
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.14, wspace=0.28)
S.save_and_show(fig, "fig_s4f_xspecies_dpsi_neuron", figure=FIG)

In [ ]:
# S4b - cross-lineage EXC vs INH pseudotime Delta-psi
ce = pd.read_csv(
    KB / "composite_ranking" / "composite_ranking_optimized2_excitatory.csv"
)[["gene_id", "gene_name", "padj", "delta_psi", "composite_score_abs", "rho_W"]]
ci = pd.read_csv(
    KB / "composite_ranking" / "composite_ranking_optimized2_inhibitory.csv"
)[["gene_id", "gene_name", "padj", "delta_psi", "composite_score_abs", "rho_W"]]

# top volcano hits per lineage (same criterion as fig2c: padj<0.05, |rho_W|>0.2, top-4 composite)
_top_exc = set(
    ce[(ce["padj"] < 0.05) & (ce["rho_W"].abs() > 0.2)].nlargest(
        4, "composite_score_abs"
    )["gene_name"]
)
_top_inh = set(
    ci[(ci["padj"] < 0.05) & (ci["rho_W"].abs() > 0.2)].nlargest(
        4, "composite_score_abs"
    )["gene_name"]
)
_top_any = _top_exc | _top_inh

m = ce[["gene_id", "gene_name", "padj", "delta_psi"]].merge(
    ci[["gene_id", "padj", "delta_psi"]], on="gene_id", suffixes=("_exc", "_inh")
)
sig = (m.padj_exc < 0.05) | (m.padj_inh < 0.05)
both = (m.padj_exc < 0.05) & (m.padj_inh < 0.05)
d = m[sig].dropna(subset=["delta_psi_exc", "delta_psi_inh"])
conc = 100 * (np.sign(d.delta_psi_exc) == np.sign(d.delta_psi_inh)).mean()
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 800 / 300))
ax.scatter(
    d.delta_psi_exc,
    d.delta_psi_inh,
    s=5,
    c="#999999",
    alpha=0.4,
    rasterized=True,
    linewidths=0,
)
db = m[both].dropna(subset=["delta_psi_exc", "delta_psi_inh"])
ax.scatter(
    db.delta_psi_exc,
    db.delta_psi_inh,
    s=9,
    c="#6e4b9e",
    alpha=0.7,
    linewidths=0,
    label=f"sig in both (n={len(db)})",
)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
lim = 1.0
ax.plot([-lim, lim], [-lim, lim], color="0.6", lw=0.5, ls="--")
ax.set_xlabel(r"excitatory  $\Delta\psi$", fontsize=S.FONT_LABEL, color=EXC_C)
ax.set_ylabel(r"inhibitory  $\Delta\psi$", fontsize=S.FONT_LABEL, color=INH_C)
ax.set_title(
    f"cross-lineage: {conc:.0f}% same-sign (n={len(d):,} sig in either)",
    fontsize=S.FONT_ANNOT,
    pad=3,
)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper left")
ax.tick_params(labelsize=S.FONT_TICK)
ax.spines[["top", "right"]].set_visible(False)
hits = d[d["gene_name"].isin(_top_any)].dropna(
    subset=["delta_psi_exc", "delta_psi_inh"]
)
for _, r in hits.iterrows():
    ax.scatter(
        r.delta_psi_exc,
        r.delta_psi_inh,
        s=20,
        c="#6e4b9e",
        linewidths=0.6,
        edgecolors="k",
        zorder=5,
    )
    ax.annotate(
        str(r.gene_name),
        (r.delta_psi_exc, r.delta_psi_inh),
        fontsize=S.FONT_ANNOT - 1,
        fontstyle="italic",
        xytext=(3, 3),
        textcoords="offset points",
    )
fig.subplots_adjust(left=0.15, right=0.96, top=0.91, bottom=0.13)
S.save_and_show(fig, "fig_s4b_cross_lineage_dpsi", figure=FIG)

In [ ]:
# S4c - per-gene W-vs-expression coupling (rho), per lineage
fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, 760 / 300))
for ax, (lin, lab) in zip(axes, LINS):
    w = pd.read_csv(
        KB
        / "correlation_analyses_expression"
        / f"utr_W_expression_spearman_per_gene_unsmoothed_{lin}.csv"
    )
    w = w[w["padj_BH"] < 0.05]  # genes with significant W-vs-expression coupling
    e = w["rho_W_expr"].dropna()
    npos = int((e > 0).sum())
    nneg = int((e < 0).sum())
    # shared bins so positive and negative bars have identical widths
    lim = e.abs().max()
    bins = np.linspace(-lim, lim, 81)  # 80 equal-width bins, symmetric around 0
    ax.hist(
        e[e > 0],
        bins=bins,
        color="#ef6548",
        alpha=0.8,
        edgecolor="none",
        label=f"positive: {npos:,}",
    )
    ax.hist(
        e[e < 0],
        bins=bins,
        color="#3690c0",
        alpha=0.8,
        edgecolor="none",
        label=f"negative: {nneg:,}",
    )
    ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(r"$\rho$ (W vs log-expression)", fontsize=S.FONT_LABEL)
    ax.set_ylabel("genes", fontsize=S.FONT_LABEL)
    ratio = (npos / nneg) if lin == "excitatory" else (nneg / npos)
    sense = "pos/neg" if lin == "excitatory" else "neg/pos"
    ax.set_title(
        f"{lab}: {ratio:.1f}x {sense} (sig. coupling)",
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[lin],
        pad=3,
    )
    ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper left")
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.15, wspace=0.25)
S.save_and_show(fig, "fig_s4c_w_vs_expression", figure=FIG)

In [ ]:
# S4e - cross-species neuron-bridge per-gene scatter
from pathlib import Path
from scipy.stats import spearmanr

OUTB = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_neuron_bridge"
)
orth = pd.read_csv(OUTB / "cross_species_neuron_orthologs.csv")
scd = orth.dropna(subset=["dpsi_qn", "dpsi_org_exc"]).copy()
rho = spearmanr(scd.dpsi_qn, scd.dpsi_org_exc)[0]
scd["conc"] = np.sign(scd.dpsi_qn) == np.sign(scd.dpsi_org_exc)
scd["mag"] = scd.dpsi_qn.abs() + scd.dpsi_org_exc.abs()
EXC_COL = S.paper_palette_lineage["excitatory"]
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 760 / 300))
ax.scatter(
    scd.loc[scd.conc, "dpsi_qn"],
    scd.loc[scd.conc, "dpsi_org_exc"],
    s=7,
    c=EXC_COL,
    alpha=0.7,
    linewidths=0,
)
ax.scatter(
    scd.loc[~scd.conc, "dpsi_qn"],
    scd.loc[~scd.conc, "dpsi_org_exc"],
    s=6,
    c="#cccccc",
    alpha=0.6,
    linewidths=0,
)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlim(scd.dpsi_qn.min() - 0.05, scd.dpsi_qn.max() + 0.18)
for _, r in scd[scd.conc].sort_values("mag", ascending=False).head(6).iterrows():
    ax.annotate(
        str(r.human_name),
        (r.dpsi_qn, r.dpsi_org_exc),
        fontsize=S.FONT_ANNOT - 1,
        fontstyle="italic",
        xytext=(2, 2),
        textcoords="offset points",
    )
ax.set_xlabel(r"mouse  qNSC$\rightarrow$neuron  $\Delta\psi$", fontsize=S.FONT_LABEL)
ax.set_ylabel(r"organoid EXC  $\Delta\psi$", fontsize=S.FONT_LABEL)
ax.set_title(
    f"cross-species: Spearman {rho:+.2f}  (n={len(scd)})", fontsize=S.FONT_ANNOT, pad=3
)
ax.tick_params(labelsize=S.FONT_TICK)
ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.16, right=0.96, top=0.91, bottom=0.14)
S.save_and_show(fig, "fig_s4e_neuron_bridge_scatter", figure=FIG)

## S4c size variants (quarter-height, full/half width)


In [ ]:
# --- S4c size variants: quarter-height full/half width ---
from matplotlib.ticker import MaxNLocator


def _build_s4c(figsize, half):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        w = pd.read_csv(
            KB
            / "correlation_analyses_expression"
            / f"utr_W_expression_spearman_per_gene_unsmoothed_{lin}.csv"
        )
        w = w[w["padj_BH"] < 0.05]
        e = w["rho_W_expr"].dropna()
        npos = int((e > 0).sum())
        nneg = int((e < 0).sum())
        lim = e.abs().max()
        bins = np.linspace(-lim, lim, 61 if half else 81)
        ax.hist(
            e[e > 0],
            bins=bins,
            color="#ef6548",
            alpha=0.8,
            edgecolor="none",
            label=f"pos: {npos:,}",
        )
        ax.hist(
            e[e < 0],
            bins=bins,
            color="#3690c0",
            alpha=0.8,
            edgecolor="none",
            label=f"neg: {nneg:,}",
        )
        ax.axvline(0, color="k", lw=0.5)
        ax.set_xlabel(r"$\rho$ (W vs expr)", fontsize=S.FONT_LABEL)
        if ax is axes[0]:
            ax.set_ylabel("genes", fontsize=S.FONT_LABEL)
        ratio = (npos / nneg) if lin == "excitatory" else (nneg / npos)
        sense = "pos/neg" if lin == "excitatory" else "neg/pos"
        ax.set_title(
            f"{lab.split()[0][:3].upper()}: {ratio:.1f}x {sense}",
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[lin],
            pad=2,
        )
        ax.legend(
            fontsize=S.FONT_LEGEND,
            frameon=False,
            loc="upper left",
            handlelength=0.9,
            handletextpad=0.4,
            borderpad=0.1,
        )
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.tick_params(labelsize=S.FONT_TICK)
        ax.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(
        left=0.06 if not half else 0.135,
        right=0.98,
        top=0.90,
        bottom=0.17,
        wspace=0.20 if not half else 0.30,
    )
    return fig


S.save_and_show(
    _build_s4c((S.FULL_W_IN, S.QUARTER_H_IN), False),
    "fig_s4c_w_vs_expression__quarter_full",
    figure=FIG,
)
S.save_and_show(
    _build_s4c((S.HALF_W_IN, S.QUARTER_H_IN), True),
    "fig_s4c_w_vs_expression__quarter_half",
    figure=FIG,
)

## S4c zoom variant (quarter-half, x in +/-0.5, dashed 0 line)


In [ ]:
# --- S4c zoom variant: quarter-half, x clipped to +/-0.5, dashed zero line (emphasise bimodal) ---
from matplotlib.ticker import MaxNLocator


def _build_s4c_zoom(figsize):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        w = pd.read_csv(
            KB
            / "correlation_analyses_expression"
            / f"utr_W_expression_spearman_per_gene_unsmoothed_{lin}.csv"
        )
        w = w[w["padj_BH"] < 0.05]
        e = w["rho_W_expr"].dropna()
        npos = int((e > 0).sum())
        nneg = int((e < 0).sum())
        lim = e.abs().max()
        bins = np.linspace(-lim, lim, 81)
        ax.hist(
            e[e > 0],
            bins=bins,
            color="#ef6548",
            alpha=0.8,
            edgecolor="none",
            label=f"pos: {npos:,}",
        )
        ax.hist(
            e[e < 0],
            bins=bins,
            color="#3690c0",
            alpha=0.8,
            edgecolor="none",
            label=f"neg: {nneg:,}",
        )
        ax.axvline(
            0, color="k", lw=0.7, ls="--"
        )  # dashed 0 line -> emphasise the two peaks
        ax.set_xlim(-0.5, 0.5)
        ax.set_xlabel(r"$\rho$ (W vs expr)", fontsize=S.FONT_LABEL)
        if ax is axes[0]:
            ax.set_ylabel("genes", fontsize=S.FONT_LABEL)
        ratio = (npos / nneg) if lin == "excitatory" else (nneg / npos)
        sense = "pos/neg" if lin == "excitatory" else "neg/pos"
        ax.set_title(
            f"{lab.split()[0][:3].upper()}: {ratio:.1f}x {sense}",
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[lin],
            pad=2,
        )
        ax.legend(
            fontsize=S.FONT_LEGEND,
            frameon=False,
            loc="upper left",
            handlelength=0.9,
            handletextpad=0.4,
            borderpad=0.1,
        )
        ax.set_xticks([-0.5, -0.25, 0, 0.25, 0.5])
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.tick_params(labelsize=S.FONT_TICK)
        ax.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(left=0.135, right=0.98, top=0.90, bottom=0.17, wspace=0.30)
    return fig


S.save_and_show(
    _build_s4c_zoom((S.HALF_W_IN, S.QUARTER_H_IN)),
    "fig_s4c_w_vs_expression__quarter_half_zoom",
    figure=FIG,
)

### S4c follow-up: shared-gene W-vs-expression coupling, EXC vs INH (sign-flip test)


In [ ]:
# --- S4c follow-up: W-vs-expression sign of the shared genes, EXC vs INH ---
# s4c plots the significant-coupling genes per lineage separately; here we intersect the two
# significant sets (padj_BH<0.05 in both lineages) and compare rho_W_expr EXC-vs-INH to test
# whether the EXC-positive / INH-negative contrast is the same genes flipping sign (it is not:
# the shared core is highly concordant; the lineage contrast is carried by lineage-specific genes).
from scipy.stats import spearmanr as _spr

_we = {
    lin: pd.read_csv(
        KB
        / "correlation_analyses_expression"
        / f"utr_W_expression_spearman_per_gene_unsmoothed_{lin}.csv"
    )
    for lin, _ in LINS
}
_e = _we["excitatory"][["gene_id", "gene_name", "rho_W_expr", "padj_BH"]].rename(
    columns={"rho_W_expr": "rho_exc", "padj_BH": "padj_exc"}
)
_i = _we["inhibitory"][["gene_id", "rho_W_expr", "padj_BH"]].rename(
    columns={"rho_W_expr": "rho_inh", "padj_BH": "padj_inh"}
)
_m = _e.merge(_i, on="gene_id", how="inner")
sh = (
    _m[(_m.padj_exc < 0.05) & (_m.padj_inh < 0.05)]
    .dropna(subset=["rho_exc", "rho_inh"])
    .copy()
)
_conc = np.sign(sh.rho_exc) == np.sign(sh.rho_inh)
_rs = _spr(sh.rho_exc, sh.rho_inh)
pp = int(((sh.rho_exc > 0) & (sh.rho_inh > 0)).sum())
nn = int(((sh.rho_exc < 0) & (sh.rho_inh < 0)).sum())

fig, ax = plt.subplots(figsize=(S.HALF_W_IN, S.HALF_W_IN * 0.85))
lim = max(sh.rho_exc.abs().max(), sh.rho_inh.abs().max()) * 1.05
ax.axhline(0, color="0.5", lw=0.5, ls="--", zorder=1)
ax.axvline(0, color="0.5", lw=0.5, ls="--", zorder=1)
ax.plot([-lim, lim], [-lim, lim], color="0.8", lw=0.6, zorder=1)
ax.scatter(
    sh.rho_exc[_conc],
    sh.rho_inh[_conc],
    s=4,
    c="#4d4d4d",
    alpha=0.45,
    edgecolors="none",
    zorder=2,
    label=f"same sign ({int(_conc.sum())})",
)
ax.scatter(
    sh.rho_exc[~_conc],
    sh.rho_inh[~_conc],
    s=7,
    c="#d01c8b",
    alpha=0.85,
    edgecolors="none",
    zorder=3,
    label=f"sign flip ({int((~_conc).sum())})",
)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect("equal")
ax.set_xlabel(
    r"$\rho$ (W vs expr), EXC",
    fontsize=S.FONT_LABEL,
    color=S.paper_palette_lineage["excitatory"],
)
ax.set_ylabel(
    r"$\rho$ (W vs expr), INH",
    fontsize=S.FONT_LABEL,
    color=S.paper_palette_lineage["inhibitory"],
)
ax.tick_params(labelsize=S.FONT_TICK)
ax.legend(
    fontsize=S.FONT_LEGEND - 0.5,
    frameon=False,
    loc="upper left",
    handletextpad=0.2,
    borderpad=0.1,
)
ax.text(
    0.97,
    0.03,
    f"++ {pp}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=S.FONT_ANNOT - 1,
    color="0.4",
)
ax.text(
    0.03,
    0.03,
    f"-- {nn}",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=S.FONT_ANNOT - 1,
    color="0.4",
)
ax.set_title(
    f"Shared W-expr-coupled genes (n={len(sh)}): concordant across lineages\n"
    rf"$\rho_s$={_rs.statistic:.2f}; {100*_conc.mean():.0f}% same sign",
    fontsize=S.FONT_ANNOT - 0.5,
    pad=3,
)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.subplots_adjust(left=0.16, right=0.97, top=0.86, bottom=0.15)
S.save_and_show(fig, "fig_s4c_shared_wexpr_exc_vs_inh", figure=FIG)
print(
    f"shared sig-in-both: {len(sh)} | same-sign {int(_conc.sum())} ({100*_conc.mean():.0f}%) | "
    f"flips {int((~_conc).sum())} | Spearman {_rs.statistic:.3f}"
)

### S4c chi-sq&perm DTU variants: histogram + shared EXC/INH scatter (double-significant)


In [ ]:
# --- chi-sq & permutation double-significant DTU variants of S4c ---
# Restrict to genes whose 3'UTR change over pseudotime is significant by both the chi-sq LRT
# (padj_chisq<0.05) and the permutation null (padj_perm<0.05), then look at their W-vs-expression
# coupling. (Base S4c set = genes with significant W-vs-expr coupling; different question.)
from matplotlib.ticker import MaxNLocator as _MNL
from scipy.stats import spearmanr as _spr2
_INTER = {}
for _lin, _ in LINS:
    _cr = pd.read_csv(KB / "composite_ranking" / f"composite_ranking_optimized2_{_lin}.csv")
    _permf = (KB / "dirmult_results" / _lin / "permutation_batch_number"
              / f"dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_{_lin}_cov-batch_number.csv")
    _pm = pd.read_csv(_permf)[["utr_name", "p_value_perm"]]
    _m = _cr.merge(_pm, left_on="gene_id", right_on="utr_name", how="left")
    _m["padj_perm"] = S.benjamini_hochberg(_m["p_value_perm"].values)
    _INTER[_lin] = set(_m.loc[(_m["padj_chisq"] < 0.05) & (_m["padj_perm"] < 0.05), "gene_id"])
    print(f"{_lin}: chisq&perm DTU-significant = {len(_INTER[_lin])}")

def _read_we(lin):
    return pd.read_csv(KB / "correlation_analyses_expression"
                       / f"utr_W_expression_spearman_per_gene_unsmoothed_{lin}.csv")

# --- S4c histogram on the chi-sq & perm DTU set (quarter-half) ---
def _build_s4c_inter(figsize):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, (lin, lab) in zip(axes, LINS):
        w = _read_we(lin)
        w = w[w.gene_id.isin(_INTER[lin])]
        e = w["rho_W_expr"].dropna()
        npos = int((e > 0).sum()); nneg = int((e < 0).sum())
        lim = e.abs().max(); bins = np.linspace(-lim, lim, 61)
        ax.hist(e[e > 0], bins=bins, color="#ef6548", alpha=0.8, edgecolor="none", label=f"pos: {npos:,}")
        ax.hist(e[e < 0], bins=bins, color="#3690c0", alpha=0.8, edgecolor="none", label=f"neg: {nneg:,}")
        ax.axvline(0, color="k", lw=0.5)
        ax.set_xlabel(r"$\rho$ (W vs expr)", fontsize=S.FONT_LABEL)
        if ax is axes[0]:
            ax.set_ylabel("genes", fontsize=S.FONT_LABEL)
        rtxt = (f"{npos/max(nneg,1):.1f}x pos/neg" if npos >= nneg else f"{nneg/max(npos,1):.1f}x neg/pos")
        ax.set_title(f"{lab.split()[0][:3].upper()}: {rtxt}", fontsize=S.FONT_ANNOT,
                     color=S.paper_palette_lineage[lin], pad=2)
        ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper left",
                  handlelength=0.9, handletextpad=0.4, borderpad=0.1)
        ax.xaxis.set_major_locator(_MNL(nbins=5)); ax.yaxis.set_major_locator(_MNL(nbins=4))
        ax.tick_params(labelsize=S.FONT_TICK); ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("chi-sq & perm DTU-significant genes", fontsize=S.FONT_ANNOT - 0.5, y=0.99)
    fig.subplots_adjust(left=0.135, right=0.98, top=0.80, bottom=0.17, wspace=0.30)
    return fig
S.save_and_show(_build_s4c_inter((S.HALF_W_IN, S.QUARTER_H_IN)),
                "fig_s4c_w_vs_expression__quarter_half_chisqperm", figure=FIG)

# --- Shared EXC-vs-INH rho scatter, restricted to double-sig-in-both genes ---
_both = _INTER["excitatory"] & _INTER["inhibitory"]
_e = _read_we("excitatory")[["gene_id", "gene_name", "rho_W_expr"]].rename(columns={"rho_W_expr": "rho_exc"})
_i = _read_we("inhibitory")[["gene_id", "rho_W_expr"]].rename(columns={"rho_W_expr": "rho_inh"})
sh = _e.merge(_i, on="gene_id", how="inner")
sh = sh[sh.gene_id.isin(_both)].dropna(subset=["rho_exc", "rho_inh"]).copy()
_conc = np.sign(sh.rho_exc) == np.sign(sh.rho_inh)
_rs = _spr2(sh.rho_exc, sh.rho_inh)
pp = int(((sh.rho_exc > 0) & (sh.rho_inh > 0)).sum()); nn = int(((sh.rho_exc < 0) & (sh.rho_inh < 0)).sum())
fig, ax = plt.subplots(figsize=(S.HALF_W_IN, S.HALF_W_IN * 0.85))
lim = max(sh.rho_exc.abs().max(), sh.rho_inh.abs().max()) * 1.05
ax.axhline(0, color="0.5", lw=0.5, ls="--"); ax.axvline(0, color="0.5", lw=0.5, ls="--")
ax.plot([-lim, lim], [-lim, lim], color="0.8", lw=0.6)
ax.scatter(sh.rho_exc[_conc], sh.rho_inh[_conc], s=9, c="#4d4d4d", alpha=0.6, edgecolors="none", label=f"same sign ({int(_conc.sum())})")
ax.scatter(sh.rho_exc[~_conc], sh.rho_inh[~_conc], s=12, c="#d01c8b", alpha=0.9, edgecolors="none", label=f"sign flip ({int((~_conc).sum())})")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
ax.set_xlabel(r"$\rho$ (W vs expr), EXC", fontsize=S.FONT_LABEL, color=S.paper_palette_lineage["excitatory"])
ax.set_ylabel(r"$\rho$ (W vs expr), INH", fontsize=S.FONT_LABEL, color=S.paper_palette_lineage["inhibitory"])
ax.tick_params(labelsize=S.FONT_TICK)
ax.legend(fontsize=S.FONT_LEGEND - 0.5, frameon=False, loc="upper left", handletextpad=0.2, borderpad=0.1)
ax.text(0.97, 0.03, f"++ {pp}", transform=ax.transAxes, ha="right", va="bottom", fontsize=S.FONT_ANNOT - 1, color="0.4")
ax.text(0.03, 0.03, f"-- {nn}", transform=ax.transAxes, ha="left", va="bottom", fontsize=S.FONT_ANNOT - 1, color="0.4")
ax.set_title(f"chi-sq & perm double-sig genes (n={len(sh)}): {100*_conc.mean():.0f}% same sign\n"
             fr"$\rho_s$={_rs.statistic:.2f}", fontsize=S.FONT_ANNOT - 0.5, pad=3)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.subplots_adjust(left=0.16, right=0.97, top=0.86, bottom=0.15)
S.save_and_show(fig, "fig_s4c_shared_wexpr_exc_vs_inh_chisqperm", figure=FIG)
print(f"double-sig both: n={len(sh)} | same-sign {int(_conc.sum())} ({100*_conc.mean():.0f}%) | flips {int((~_conc).sum())} | rho_s {_rs.statistic:.3f}")
